# Role-Aware Weighted Semantic Matching (RAWSM)
A fairness-aware CV screening system that ranks candidates against role-specific job descriptions using section-level semantic similarity.

## 1. Setup
Install dependencies and import libraries.

In [1]:
!pip install PyMuPDF pandas numpy scikit-learn sentence-transformers fairlearn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 16.0 MB/s eta 0:00:00


In [2]:
import fitz
import os
import re
import pandas as pd
import numpy as np
import torch
from sentence_transformers import util
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_difference
from sklearn.dummy import DummyClassifier
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

## 2. Configuration
Mount Google Drive and set the path to the CV folder.

In [3]:
drive.mount('/content/drive')

# ← Change this to your actual folder path in Drive
CV_FOLDER = '/content/drive/MyDrive/I-Hire-RAs-Candidates'

Mounted at /content/drive


## 3. Load Embedding Model
Using `all-MiniLM-L6-v2` for fast, high-quality sentence embeddings.

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.


## 4. Job Descriptions
Define the target role descriptions that CVs will be matched against.

In [5]:
JD_AI = """
We are seeking an AI Engineer to design and develop intelligent systems
for candidate assessment and job matching using multimodal data.
The role involves building AI models for resume understanding using NLP,
analyzing interviews through text, audio, and video modalities,
and designing candidate ranking and scoring systems.
The engineer will implement LLM-based pipelines including RAG and prompt engineering,
apply fairness-aware and bias mitigation techniques,
and build explainable AI components for decision transparency using tools like LIME and SHAP.
Strong Python programming is required along with deep understanding of
machine learning, deep learning, transformers, embeddings, and semantic similarity.
Experience with PyTorch or TensorFlow, Hugging Face frameworks, and model deployment is expected.
The ideal candidate understands model evaluation metrics and can optimize
systems for performance and scalability.
"""

JD_FS = """
We are seeking a Full Stack Engineer to develop scalable systems
for an AI-powered recruitment platform.
Responsibilities include developing and maintaining frontend interfaces using React,
building backend APIs with Node.js, Django, or FastAPI,
and integrating AI models into production systems.
The engineer will design and manage both structured databases like PostgreSQL
and unstructured databases like MongoDB and vector databases.
Building scalable pipelines for CV processing and candidate data management is required,
along with ensuring system performance, security, and reliability.
Strong experience with JavaScript, TypeScript, REST APIs, and database design is required.
Familiarity with Docker, cloud deployment on AWS or Azure,
vector search systems, and real-time scalable pipelines is preferred.
Understanding of software architecture and system design principles is essential.
"""

## 5. CV Extraction
Extract text from PDF CVs. Files exceeding the page or word thresholds are flagged as bulk/merged PDFs and excluded from scoring.

In [6]:
def extract_text(path):
    try:
        doc     = fitz.open(path)
        text    = " ".join(page.get_text() for page in doc)
        n_pages = len(doc)
        doc.close()
        return text.strip(), n_pages
    except:
        return "", 0

# ── Bulk-PDF detection thresholds ────────────────────────────────────────────
# A single candidate CV almost never exceeds 5 pages or 3,000 words.
# Files above either threshold are treated as bulk / merged PDFs and excluded
# from all scoring to prevent one document from dominating the ranked lists.
BULK_PAGE_THRESHOLD = 5
BULK_WORD_THRESHOLD = 3000

records = []
for fname in sorted(os.listdir(CV_FOLDER)):
    if fname.lower().endswith('.pdf'):
        path              = os.path.join(CV_FOLDER, fname)
        text, n_pages     = extract_text(path)
        word_count        = len(text.split())
        is_bulk           = (n_pages > BULK_PAGE_THRESHOLD) or (word_count > BULK_WORD_THRESHOLD)
        records.append({
            'filename'  : fname,
            'name'      : fname.replace('.pdf', '').replace('_', ' ').strip(),
            'raw_text'  : text,
            'char_count': len(text),
            'word_count': word_count,
            'n_pages'   : n_pages,
            'bulk_flag' : is_bulk,
        })

df = pd.DataFrame(records)

print(f"Extracted {len(df)} PDFs total")
print(f"Word count range : {df['word_count'].min()} — {df['word_count'].max()}")
print(f"Page count range : {df['n_pages'].min()} — {df['n_pages'].max()}")
print(f"Empty extractions: {(df['char_count'] < 100).sum()}")
print()

bulk = df[df['bulk_flag']]
if len(bulk):
    print(f"⚠  Bulk PDFs detected ({len(bulk)}) — excluded from all scoring:")
    for _, r in bulk.iterrows():
        print(f"   • {r['filename']}  ({r['n_pages']} pages, {r['word_count']:,} words)")
else:
    print("✓ No bulk PDFs detected.")
print()
print(f"Individual CVs available for scoring: {(~df['bulk_flag']).sum()}")

MuPDF error: syntax error: unknown keyword: 'Qq'

MuPDF error: syntax error: unknown keyword: 'Qq'

Extracted 87 PDFs total
Word count range : 175 — 23710
Page count range : 1 — 63
Empty extractions: 0

⚠  Bulk PDFs detected (2) — excluded from all scoring:
   • All CVS - I-hire.pdf  (63 pages, 23,710 words)
   • Ibrahim Mamdouh Khafagy.pdf  (7 pages, 1,424 words)

Individual CVs available for scoring: 85


## 6. Text Preprocessing
Normalize technical abbreviations, strip PII (emails, phone numbers, URLs), and clean formatting artifacts.

In [7]:
# ── Technical term normalization dictionary ──────────────────
# Maps common abbreviations and variants to a standard form
# so the embedding model sees consistent terminology
TECH_NORMALIZE = {
    r'\bNLP\b'          : 'natural language processing',
    r'\bML\b'           : 'machine learning',
    r'\bDL\b'           : 'deep learning',
    r'\bCV\b'           : 'computer vision',
    r'\bLLM\b'          : 'large language model',
    r'\bRAG\b'          : 'retrieval augmented generation',
    r'\bXAI\b'          : 'explainable artificial intelligence',
    r'\bAPI\b'          : 'application programming interface',
    r'\bCI/CD\b'        : 'continuous integration continuous deployment',
    r'\bDBMS\b'         : 'database management system',
    r'\bOOP\b'          : 'object oriented programming',
    r'\bREST\b'         : 'representational state transfer',
    r'\bMLOps\b'        : 'machine learning operations',
    r'\bFE\b'           : 'frontend',
    r'\bBE\b'           : 'backend',
}

def preprocess(text):
    if not text or len(text.strip()) < 10:
        return ""

    # ── Step 1: Fix encoding artifacts ───────────────────────
    text = text.encode('utf-8', errors='ignore').decode('utf-8')
    # Remove common PDF garbled characters
    text = re.sub(r'[â€™â€œâ€\x80-\x9f]', ' ', text)
    # Fix hyphenated line breaks (e.g. "experi-\nence" → "experience")
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)

    # ── Step 2: Remove personal information noise ─────────────
    # Email addresses
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    # Phone numbers (various formats)
    text = re.sub(r'(\+?\d[\d\s\-().]{7,}\d)', '', text)
    # URLs and web addresses
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # LinkedIn and GitHub handles
    text = re.sub(r'linkedin\.com/\S+', '', text)
    text = re.sub(r'github\.com/\S+', '', text)

    # ── Step 3: Normalize whitespace and punctuation ──────────
    # Replace bullet symbols with space
    text = re.sub(r'[•●◦▪▸►✓✔–—]', ' ', text)
    # Remove page numbers (standalone numbers on a line)
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)
    # Collapse multiple spaces and newlines
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Remove lines that are just punctuation or symbols
    text = re.sub(r'^\s*[|/\\=_*#~`]+\s*$', '', text, flags=re.MULTILINE)

    # ── Step 4: Normalize technical terminology ───────────────
    for pattern, replacement in TECH_NORMALIZE.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    return text.strip()

# Apply to all CVs
df['clean_text'] = df['raw_text'].apply(preprocess)

# Verify improvement
df['clean_word_count'] = df['clean_text'].apply(lambda t: len(t.split()))

print("Pre-processing complete.")
print(f"Average word count before: {df['word_count'].mean():.0f}")
print(f"Average word count after : {df['clean_word_count'].mean():.0f}")
print(f"Difference represents noise removed per CV on average: "
      f"{(df['word_count'] - df['clean_word_count']).mean():.0f} words")

Pre-processing complete.
Average word count before: 985
Average word count after : 956
Difference represents noise removed per CV on average: 29 words


## 7. Section Detection
Segment each CV into structured sections (Experience, Skills, Projects, etc.) to enable section-level weighted scoring.

In [8]:
# Section header patterns — covers most common CV formats
SECTION_PATTERNS = {
    'summary'       : r'(summary|objective|profile|about me|personal statement)',
    'experience'    : r'(experience|employment|work history|professional background|career)',
    'projects'      : r'(projects|portfolio|personal projects|key projects|academic projects)',
    'skills'        : r'(skills|technical skills|core competencies|technologies|tech stack|tools)',
    'education'     : r'(education|academic|qualifications|degrees|university|college)',
    'certifications': r'(certifications|certificates|courses|training|licenses|credentials)',
}

def detect_sections(text):
    """
    Split CV text into sections based on common headers.
    Returns a dict of section_name -> section_text.
    Unmatched text goes into 'general'.
    """
    sections = {k: '' for k in SECTION_PATTERNS.keys()}
    sections['general'] = ''

    if not text:
        return sections

    lines      = text.split('\n')
    current    = 'general'
    buffer     = []

    for line in lines:
        line_stripped = line.strip()
        matched       = False

        # Check if this line is a section header
        for section, pattern in SECTION_PATTERNS.items():
            if re.match(pattern, line_stripped, re.IGNORECASE) and len(line_stripped) < 60:
                # Save previous buffer to current section
                if buffer:
                    sections[current] += ' '.join(buffer) + ' '
                    buffer = []
                current = section
                matched = True
                break

        if not matched and line_stripped:
            buffer.append(line_stripped)

    # Save remaining buffer
    if buffer:
        sections[current] += ' '.join(buffer)

    # Clean each section
    for key in sections:
        sections[key] = sections[key].strip()

    return sections

# Apply section detection to all CVs
df['sections'] = df['clean_text'].apply(detect_sections)

# Quick audit — how well did detection work?
section_coverage = pd.DataFrame([
    {
        'name': row['name'],
        **{k: len(v) > 20 for k, v in row['sections'].items()}
    }
    for _, row in df.iterrows()
])

print("Section Detection Coverage (% of CVs where section was found):")
print("-" * 50)
for col in list(SECTION_PATTERNS.keys()) + ['general']:
    pct = section_coverage[col].mean() * 100
    print(f"  {col:<20}: {pct:.1f}%")

Section Detection Coverage (% of CVs where section was found):
--------------------------------------------------
  summary             : 54.0%
  experience          : 48.3%
  projects            : 82.8%
  skills              : 94.3%
  education           : 98.9%
  certifications      : 63.2%
  general             : 95.4%


## 8. Baseline Semantic Scoring
Compute whole-document cosine similarity as a baseline, with ambiguity, missing-data, and bias flags for quality control.

In [9]:
# ── SEMANTIC EMBEDDINGS ───────────────────────────────────────
print("Encoding CVs... (this may take a minute)")

# Encode all CVs and both JDs
cv_texts    = df['clean_text'].tolist()
cv_embeddings = model.encode(cv_texts, show_progress_bar=True)
jd_ai_emb   = model.encode([JD_AI])
jd_fs_emb   = model.encode([JD_FS])

# Cosine similarity scores
df['score_ai'] = cosine_similarity(cv_embeddings, jd_ai_emb).flatten()
df['score_fs'] = cosine_similarity(cv_embeddings, jd_fs_emb).flatten()

# ── CRITERIA 1: AMBIGUITY ─────────────────────────────────────
# Short CVs give the model too little context to embed meaningfully.
# The score may look reasonable but is unreliable.
df['ambiguity_flag'] = df['word_count'] < 200
df['ambiguity_note'] = df['ambiguity_flag'].apply(
    lambda x: "⚠ Too short — embedding unreliable" if x else "OK"
)

# ── CRITERIA 2: MISSING DATA ──────────────────────────────────
# Extraction failed or CV is essentially empty.
# Scores for these are meaningless and should be excluded.
df['missing_flag'] = df['char_count'] < 100
df['missing_note'] = df['missing_flag'].apply(
    lambda x: "⚠ Extraction failed — exclude from ranking" if x else "OK"
)

# ── CRITERIA 3: BIAS IN RANKING ───────────────────────────────
# Semantic models can still reflect bias.
# We flag CVs that score suspiciously high on one role
# while scoring near zero on the other — this can indicate
# that generic soft-skill language inflated the score
# rather than genuine technical alignment.
df['score_gap']  = abs(df['score_ai'] - df['score_fs'])
df['bias_flag']  = (
    ((df['score_ai'] > 0.35) & (df['score_fs'] < 0.10)) |
    ((df['score_fs'] > 0.35) & (df['score_ai'] < 0.10))
)
df['bias_note'] = df['bias_flag'].apply(
    lambda x: "⚠ Extreme score gap — manual review advised" if x else "OK"
)

# ── CRITERIA 4: EXPLAINABILITY ────────────────────────────────
# Since embeddings are not interpretable on their own,
# we extract the top semantically relevant sentences from
# each CV by comparing sentence-level embeddings to the JD.
# This tells the recruiter WHY the candidate ranked where they did.

def top_sentences(cv_text, jd_embedding, n=2):
    """Find the n sentences in the CV most semantically similar to the JD."""
    # Split into sentences
    sentences = [s.strip() for s in re.split(r'[.\n]', cv_text) if len(s.strip()) > 30]
    if not sentences:
        return "No extractable evidence found"

    sent_embeddings = model.encode(sentences)
    sims = cosine_similarity(sent_embeddings, jd_embedding).flatten()
    top_idx = sims.argsort()[::-1][:n]

    result = []
    for idx in top_idx:
        short = sentences[idx][:120]  # trim long sentences for readability
        result.append(f'"{short}..." ({sims[idx]:.2f})')
    return " | ".join(result)

print("Extracting evidence sentences for explainability...")
df['explain_ai'] = df['clean_text'].apply(lambda t: top_sentences(t, jd_ai_emb))
df['explain_fs'] = df['clean_text'].apply(lambda t: top_sentences(t, jd_fs_emb))

Encoding CVs... (this may take a minute)


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Extracting evidence sentences for explainability...


In [10]:
# Exclude missing CVs from rankings
clean_df = df[~df['missing_flag']].copy()

rank_ai = clean_df.sort_values('score_ai', ascending=False).head(10).reset_index(drop=True)
rank_fs = clean_df.sort_values('score_fs', ascending=False).head(10).reset_index(drop=True)

# ── AI ENGINEER RANKING ───────────────────────────────────────
print("=" * 70)
print("TOP 10 — AI ENGINEER ROLE (Semantic Similarity)")
print("=" * 70)
for i, row in rank_ai.iterrows():
    print(f"\n#{i+1}  {row['name']}")
    print(f"    Semantic Score : {row['score_ai']:.3f}")
    print(f"    Evidence       : {row['explain_ai']}")
    print(f"    Ambiguity      : {row['ambiguity_note']}")
    print(f"    Bias Check     : {row['bias_note']}")

# ── FULL STACK RANKING ────────────────────────────────────────
print("\n")
print("=" * 70)
print("TOP 10 — FULL STACK ENGINEER ROLE (Semantic Similarity)")
print("=" * 70)
for i, row in rank_fs.iterrows():
    print(f"\n#{i+1}  {row['name']}")
    print(f"    Semantic Score : {row['score_fs']:.3f}")
    print(f"    Evidence       : {row['explain_fs']}")
    print(f"    Ambiguity      : {row['ambiguity_note']}")
    print(f"    Bias Check     : {row['bias_note']}")

# ── FLAGS SUMMARY ─────────────────────────────────────────────
print("\n")
print("=" * 70)
print("FLAGGED CVs — Require Manual Review")
print("=" * 70)
flagged = df[df['ambiguity_flag'] | df['missing_flag'] | df['bias_flag']]
print(f"\nTotal flagged: {len(flagged)} / {len(df)}\n")
for _, row in flagged.iterrows():
    print(f"  {row['name']}")
    print(f"    Ambiguity : {row['ambiguity_note']}")
    print(f"    Missing   : {row['missing_note']}")
    print(f"    Bias      : {row['bias_note']}")
    print()

TOP 10 — AI ENGINEER ROLE (Semantic Similarity)

#1  Mohamed Abden Nour El-Din
    Semantic Score : 0.678
    Evidence       : "Fresh AI/​machine learning Engineer with hands-on experience building production-ready retrieval augmented generation sy..." (0.51) | "Ai and machine learning engineer..." (0.50)
    Ambiguity      : OK
    Bias Check     : OK

#2  All CVS - I-hire
    Semantic Score : 0.649
    Evidence       : "AI Engineer specializing in large language model-based systems and applied deep learning, with hands-on experience build..." (0.62) | "AI and Data Science graduate with hands-on experience building machine learning and deep learning pipelines in Python,..." (0.60)
    Ambiguity      : OK
    Bias Check     : OK

#3  AbdallahEmam CV 2 1
    Semantic Score : 0.649
    Evidence       : "Designed a Multi-Agent System for large language model-Based Evaluation: Architected a modular AI pipeline with three sp..." (0.50) | "large language model/machine learning operations Too

## 9. RAWSM Weighted Scoring
Role-Aware Weighted Semantic Matching: score each CV section independently, apply role-specific weights, and apply a content-density penalty.

In [12]:
# ── Filter out bulk PDFs before any scoring ───────────────────────────────────
scoring_df = df[~df['bulk_flag']].copy().reset_index(drop=True)
print(f"Scoring {len(scoring_df)} individual CVs "
      f"({df['bulk_flag'].sum()} bulk PDF(s) excluded)\n")

# ── Re-apply preprocessing and section detection on scoring_df ───────────────
# Needed because scoring_df is built after extraction re-creates df,
# so clean_text and sections columns must be derived fresh here.
scoring_df['clean_text'] = scoring_df['raw_text'].apply(preprocess)
scoring_df['sections']   = scoring_df['clean_text'].apply(detect_sections)

# ── Role-specific section weights (must sum to 1.0) ───────────────────────────
ROLE_WEIGHTS = {
    'AI Engineer': {
        'summary'       : 0.10,
        'experience'    : 0.30,
        'projects'      : 0.25,
        'skills'        : 0.20,
        'education'     : 0.05,
        'certifications': 0.05,
        'general'       : 0.05,
    },
    'Full Stack Engineer': {
        'summary'       : 0.10,
        'experience'    : 0.30,
        'projects'      : 0.25,
        'skills'        : 0.20,
        'education'     : 0.05,
        'certifications': 0.05,
        'general'       : 0.05,
    },
}

JOB_DESCRIPTIONS = {
    'AI Engineer'        : JD_AI,
    'Full Stack Engineer': JD_FS,
}

jd_embeddings = {
    role: model.encode(text, convert_to_tensor=True)
    for role, text in JOB_DESCRIPTIONS.items()
}

# ── Content-density score ─────────────────────────────────────────────────────
SECTION_NAMES = list(SECTION_PATTERNS.keys())  # excludes 'general'

def content_density(sections_dict, total_words):
    """Ratio of named-section words to total words (0–1)."""
    if total_words == 0:
        return 0.0
    named_words = sum(
        len(v.split())
        for k, v in sections_dict.items()
        if k in SECTION_NAMES and v
    )
    return min(named_words / total_words, 1.0)

# ── Length-normalised section encoding ───────────────────────────────────────
MAX_SECTION_WORDS = 120

def truncate_section(text, max_words=MAX_SECTION_WORDS):
    words = text.split()
    return ' '.join(words[:max_words]) if len(words) > max_words else text

def section_level_scores(sections_dict, jd_emb):
    """Cosine similarity of each (truncated) section against the JD embedding."""
    scores = {}
    for sec_name, sec_text in sections_dict.items():
        if sec_text and len(sec_text.strip()) > 20:
            trimmed          = truncate_section(sec_text)
            sec_emb          = model.encode(trimmed, convert_to_tensor=True)
            scores[sec_name] = float(util.cos_sim(sec_emb, jd_emb))
        else:
            scores[sec_name] = 0.0
    return scores

def weighted_score(sec_scores, weights):
    """Weighted average of section scores."""
    total, total_weight = 0.0, 0.0
    for sec, w in weights.items():
        total        += sec_scores.get(sec, 0.0) * w
        total_weight += w
    return total / total_weight if total_weight else 0.0

# ── Score every individual CV × every role ────────────────────────────────────
results = []

for _, row in scoring_df.iterrows():
    density = content_density(row['sections'], row['word_count'])
    record  = {
        'filename'       : row['filename'],
        'name'           : row['name'],
        'word_count'     : row['word_count'],
        'n_pages'        : row['n_pages'],
        'content_density': density,
        'ambiguous'      : (row['word_count'] < 200) or (density < 0.25),
    }
    for role, jd_emb in jd_embeddings.items():
        sec_scores = section_level_scores(row['sections'], jd_emb)
        raw_score  = weighted_score(sec_scores, ROLE_WEIGHTS[role])
        # Density penalty: score × (0.85 + 0.15 × density)
        # A zero-density CV loses at most 15% of its raw score
        penalised  = raw_score * (0.85 + 0.15 * density)
        record[f'{role}_section_scores'] = sec_scores
        record[f'{role}_raw']            = penalised
    results.append(record)

results_df = pd.DataFrame(results)

# ── Min-Max normalisation within each role ────────────────────────────────────
scaler = MinMaxScaler()
for role in JOB_DESCRIPTIONS:
    col_raw  = f'{role}_raw'
    col_norm = f'{role}_norm'
    results_df[col_norm] = scaler.fit_transform(results_df[[col_raw]]).flatten()

# ── Summary ───────────────────────────────────────────────────────────────────
print("RAWSM Weighted + Normalised Scores — Summary")
print("-" * 55)
print(f"Content density — mean: {results_df['content_density'].mean():.2f}  "
      f"min: {results_df['content_density'].min():.2f}  "
      f"max: {results_df['content_density'].max():.2f}")
print(f"Ambiguity flags  : {results_df['ambiguous'].sum()} / {len(results_df)}")
print()
for role in JOB_DESCRIPTIONS:
    col  = f'{role}_norm'
    top3 = results_df.nlargest(3, col)[['name', col, 'content_density']]
    print(f"Top 3 — {role}:")
    for _, r in top3.iterrows():
        print(f"  {r['name']:<42} score={r[col]:.4f}  density={r['content_density']:.2f}")
    print()

Scoring 85 individual CVs (2 bulk PDF(s) excluded)

RAWSM Weighted + Normalised Scores — Summary
-------------------------------------------------------
Content density — mean: 0.89  min: 0.17  max: 1.00
Ambiguity flags  : 2 / 85

Top 3 — AI Engineer:
  AI Engineer Resume                         score=1.0000  density=1.00
  Mariam Elwakel CV                          score=0.9890  density=0.94
  Mohamed-Elmineawy-AI-1                     score=0.9814  density=0.99

Top 3 — Full Stack Engineer:
  Hatem-Salem-Resume                         score=1.0000  density=0.99
  MARK MAGED AI                              score=0.9859  density=0.92
  Mohamed Ahmed Fathy CV                     score=0.9388  density=0.88



## 10. Fairness Calibration
Use FairLearn's `ThresholdOptimizer` to audit and mitigate demographic parity differences across candidate groups.

In [17]:
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_difference, MetricFrame
from sklearn.linear_model import LogisticRegression
import numpy as np

# ── Sensitive feature: CV length tier ────────────────────────────────────────
# Tier 0 — Short  : < 200 words  (sparse CVs, high embedding uncertainty)
# Tier 1 — Medium : 200–500 words (typical one-page CVs)
# Tier 2 — Long   : > 500 words  (detailed multi-section CVs)

def length_tier(word_count):
    if word_count < 200:
        return 0
    elif word_count <= 500:
        return 1
    else:
        return 2

results_df['length_tier_raw'] = results_df['word_count'].apply(length_tier)

# ── Merge degenerate tiers ────────────────────────────────────────────────────
# ThresholdOptimizer requires each sensitive group to have ≥ 2 samples with
# both label classes present. Groups that are too small are merged upward.
MIN_GROUP_SIZE = 5

tier_counts = results_df['length_tier_raw'].value_counts()
print("Raw tier counts:")
print(tier_counts.to_string())
print()

def merge_small_tiers(tier, word_count, tier_counts, min_size=MIN_GROUP_SIZE):
    """Merge undersized tiers into the next tier up."""
    if tier_counts.get(tier, 0) < min_size:
        return min(tier + 1, 2)  # merge upward, cap at Tier 2
    return tier

results_df['length_tier'] = results_df.apply(
    lambda r: merge_small_tiers(r['length_tier_raw'], r['word_count'], tier_counts),
    axis=1
)

TIER_LABELS = {0: 'Short (<200w)', 1: 'Medium (200–500w)', 2: 'Long (>500w)'}
results_df['length_tier_label'] = results_df['length_tier'].map(TIER_LABELS)

print("Effective tier distribution (after merging small groups):")
print(results_df['length_tier_label'].value_counts().to_string())
print()

# ── FairLearn calibration ─────────────────────────────────────────────────────
fl_records = []

for role in JOB_DESCRIPTIONS:
    col_norm  = f'{role}_norm'
    X         = results_df[[col_norm]].values
    y         = (results_df[col_norm] >= 0.5).astype(int).values
    sensitive = results_df['length_tier'].values

    n_groups  = len(np.unique(sensitive))
    n_classes = len(np.unique(y))

    if n_groups < 2 or n_classes < 2:
        print(f"{role}: insufficient diversity after merging — skipping adjustment.")
        results_df[f'{role}_fl_adjusted'] = False
        fl_records.append({'role': role, 'dpd_before': None, 'dpd_after': None, 'n_adjusted': 0})
        continue

    # Validate each group has both label classes
    skip = False
    for grp in np.unique(sensitive):
        mask = sensitive == grp
        if len(np.unique(y[mask])) < 2:
            print(f"{role}: group {grp} still degenerate after merging — skipping adjustment.")
            skip = True
            break

    if skip:
        results_df[f'{role}_fl_adjusted'] = False
        fl_records.append({'role': role, 'dpd_before': None, 'dpd_after': None, 'n_adjusted': 0})
        continue

    base_clf = LogisticRegression(max_iter=500)
    base_clf.fit(X, y)

    y_pred_base = base_clf.predict(X)
    dpd_before  = demographic_parity_difference(y, y_pred_base, sensitive_features=sensitive)

    try:
        optimizer = ThresholdOptimizer(
            estimator      = base_clf,
            constraints    = 'demographic_parity',
            predict_method = 'predict_proba',
            objective      = 'balanced_accuracy_score',
        )
        optimizer.fit(X, y, sensitive_features=sensitive)
        adjusted_labels = optimizer.predict(X, sensitive_features=sensitive)
    except Exception as e:
        print(f"  ThresholdOptimizer error for {role}: {e}")
        adjusted_labels = y_pred_base

    dpd_after  = demographic_parity_difference(y, adjusted_labels, sensitive_features=sensitive)
    n_adjusted = int((adjusted_labels != y_pred_base).sum())

    results_df[f'{role}_fl_adjusted'] = (adjusted_labels != y_pred_base)

    fl_records.append({
        'role'      : role,
        'dpd_before': dpd_before,
        'dpd_after' : dpd_after,
        'n_adjusted': n_adjusted,
    })

    mf = MetricFrame(
        metrics            = {'selection_rate': lambda yt, yp: yp.mean()},
        y_true             = y,
        y_pred             = adjusted_labels,
        sensitive_features = sensitive,
    )

    print(f"{role}")
    print(f"  Demographic Parity Difference — before : {dpd_before:.4f}")
    print(f"  Demographic Parity Difference — after  : {dpd_after:.4f}")
    print(f"  Candidates adjusted by FairLearn       : {n_adjusted}")
    print(f"  Selection rate by length tier (after adjustment):")
    for tier, rate in mf.by_group['selection_rate'].items():
        label = TIER_LABELS.get(tier, str(tier))
        print(f"    {label:<22}: {rate:.3f}")
    print()

Raw tier counts:
length_tier_raw
2    67
1    17
0     1

Effective tier distribution (after merging small groups):
length_tier_label
Long (>500w)         67
Medium (200–500w)    18

AI Engineer
  Demographic Parity Difference — before : 0.0116
  Demographic Parity Difference — after  : 0.0779
  Candidates adjusted by FairLearn       : 5
  Selection rate by length tier (after adjustment):
    Medium (200–500w)     : 0.556
    Long (>500w)          : 0.478

Full Stack Engineer
  Demographic Parity Difference — before : 0.1269
  Demographic Parity Difference — after  : 0.0522
  Candidates adjusted by FairLearn       : 5
  Selection rate by length tier (after adjustment):
    Medium (200–500w)     : 0.500
    Long (>500w)          : 0.552

FairLearn calibration complete.


## 11. Evaluation — Precision@K
Evaluate ranking quality using Precision@5 and Precision@10 with a relevance threshold at the 60th percentile of raw scores.

In [14]:
# Relevance thresholds derived from baseline semantic scores
RELEVANCE_THRESHOLDS = {
    'AI Engineer'        : 0.55,
    'Full Stack Engineer': 0.45,
}

# Merge baseline scores from scoring_df if available, else use raw SAWSM as proxy
BASELINE_COLS = {
    'AI Engineer'        : 'ai_score',
    'Full Stack Engineer': 'fs_score',
}

for role, bcol in BASELINE_COLS.items():
    if bcol in scoring_df.columns:
        score_map = scoring_df.set_index('filename')[bcol].to_dict()
        results_df[f'{role}_baseline'] = results_df['filename'].map(score_map).fillna(0.0)
    else:
        results_df[f'{role}_baseline'] = results_df[f'{role}_raw']

def precision_at_k(ranked_df, score_col, relevance_col, k):
    """Fraction of top-k candidates that are truly relevant."""
    top_k = ranked_df.nlargest(k, score_col)
    return top_k[relevance_col].sum() / k

print("Precision@K Evaluation")
print("=" * 55)

eval_records = []

for role in JOB_DESCRIPTIONS:
    raw_col = f'{role}_raw'
    norm_col = f'{role}_norm'
    rel_col  = f'{role}_relevant'

    # Relevance threshold = 60th percentile of raw scores within this role
    threshold = results_df[raw_col].quantile(0.60)
    results_df[rel_col] = (results_df[raw_col] >= threshold).astype(int)

    n_relevant = results_df[rel_col].sum()
    print(f"\n{role}")
    print(f"  Raw score threshold (60th pct) : {threshold:.4f}")
    print(f"  Candidates marked relevant     : {n_relevant} / {len(results_df)}")

    for k in [5, 10]:
        p_sawsm = precision_at_k(results_df, norm_col, rel_col, k)
        eval_records.append({
            'role'     : role,
            'k'        : k,
            'threshold': round(threshold, 4),
            'P@K_sawsm': p_sawsm,
        })
        print(f"  Precision@{k}  : {p_sawsm:.3f}")

eval_df = pd.DataFrame(eval_records)
print("\n" + "=" * 55)
print(eval_df.to_string(index=False))

Precision@K Evaluation

AI Engineer
  Raw score threshold (60th pct) : 0.3270
  Candidates marked relevant     : 34 / 85
  Precision@5  : 1.000
  Precision@10  : 1.000

Full Stack Engineer
  Raw score threshold (60th pct) : 0.2686
  Candidates marked relevant     : 34 / 85
  Precision@5  : 1.000
  Precision@10  : 1.000

               role  k  threshold  P@K_sawsm
        AI Engineer  5     0.3270        1.0
        AI Engineer 10     0.3270        1.0
Full Stack Engineer  5     0.2686        1.0
Full Stack Engineer 10     0.2686        1.0


## 12. Final Rankings
Display the top-K candidates per role with section breakdowns, fairness flags, and ambiguity warnings.

In [15]:
TOP_K = 10

SECTION_ORDER = ['experience', 'projects', 'skills', 'summary',
                 'certifications', 'education', 'general']

def top_sections(sec_scores, n=3):
    """Return the n highest-scoring section names."""
    sorted_secs = sorted(sec_scores.items(), key=lambda x: x[1], reverse=True)
    return [s for s, v in sorted_secs if v > 0.0][:n]

for role in JOB_DESCRIPTIONS:
    ncol    = f'{role}_norm'
    sec_col = f'{role}_section_scores'
    fl_col  = f'{role}_fl_adjusted'
    rel_col = f'{role}_relevant'

    ranked = results_df.nlargest(TOP_K, ncol).reset_index(drop=True)

    p5  = eval_df.loc[(eval_df['role']==role) & (eval_df['k']==5),  'P@K_sawsm'].values[0]
    p10 = eval_df.loc[(eval_df['role']==role) & (eval_df['k']==10), 'P@K_sawsm'].values[0]

    sep = '=' * 70
    print(f"\n{sep}")
    print(f"TOP {TOP_K} — {role.upper()} ROLE  |  SAWSM")
    print(f"Precision@5 = {p5:.3f}   Precision@10 = {p10:.3f}")
    print(sep)

    for rank, row in ranked.iterrows():
        sec_scores  = row[sec_col]
        top_secs    = top_sections(sec_scores)
        fl_flag     = '⚑ FairLearn adjusted' if row.get(fl_col, False) else ''
        relevant_mk = '✓ Relevant' if row[rel_col] else ''

        sec_breakdown = '  |  '.join(
            f"{s}: {sec_scores.get(s, 0):.3f}"
            for s in SECTION_ORDER
            if sec_scores.get(s, 0) > 0
        )
        drivers = ', '.join(top_secs)

        print(f"\n#{rank+1:>2}  {row['name']}")
        print(f"     Final Score (normalised) : {row[ncol]:.4f}   {relevant_mk}")
        print(f"     Content Density          : {row['content_density']:.2f}")
        print(f"     Section Breakdown        : {sec_breakdown}")
        print(f"     Top Drivers              : {drivers}")
        if fl_flag:
            print(f"     {fl_flag}")
        if row['ambiguous']:
            print(f"     ⚠ Ambiguity flag: short/low-density CV ({row['word_count']} words, density={row['content_density']:.2f})")

    print()

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("PRECISION@K SUMMARY — Baseline vs SAWSM")
print("=" * 70)
print(eval_df.to_string(index=False))


TOP 10 — AI ENGINEER ROLE  |  SAWSM
Precision@5 = 1.000   Precision@10 = 1.000

# 1  AI Engineer Resume
     Final Score (normalised) : 1.0000   ✓ Relevant
     Content Density          : 1.00
     Section Breakdown        : experience: 0.608  |  projects: 0.448  |  skills: 0.457  |  summary: 0.624  |  certifications: 0.469  |  education: 0.259  |  general: 0.245
     Top Drivers              : summary, experience, certifications

# 2  Mariam Elwakel CV
     Final Score (normalised) : 0.9890   ✓ Relevant
     Content Density          : 0.94
     Section Breakdown        : experience: 0.539  |  projects: 0.612  |  skills: 0.447  |  summary: 0.568  |  education: 0.365  |  general: 0.351
     Top Drivers              : projects, summary, experience

# 3  Mohamed-Elmineawy-AI-1
     Final Score (normalised) : 0.9814   ✓ Relevant
     Content Density          : 0.99
     Section Breakdown        : experience: 0.523  |  projects: 0.527  |  skills: 0.547  |  summary: 0.556  |  certifications